In [1]:
import pandas as pd
from datetime import datetime
from tqdm.auto import tqdm

# Import RAG

In [2]:
import sys
sys.path.append('../scripts')
import rag
import vectors
import ingest

# Loading data

## Answers to synthetic questions

In [3]:
df_synth_a = pd.read_csv('../data/data-synth-answer.csv', sep='\t', dtype=str)
df_synth_a

,pmid,ollama_seed,answer_llama3.2:1b,answer_gemma3:1b
0,40255413,0,Based on the context provided by the papers fr...,"Based on the provided text, the answer is:\n\n..."
1,40255413,1,Based on the provided context and papers from ...,"Okay, here’s an answer based on the provided t..."
2,40255413,2,Based on the context provided by the papers fr...,"Okay, based on the provided context and papers..."
3,40255413,3,"Based on the provided papers from PubMed, a co...","Okay, based on the provided context and papers..."
4,40255413,4,Based on the provided context and papers from ...,"Okay, let’s analyze the provided text and answ..."
5,40264093,0,Based on the context provided by the papers yo...,"Okay, here’s an analysis of the provided text,..."
6,40264093,1,Individuals on the autism spectrum can navigat...,"Okay, let’s answer your question based on the ..."
7,40264093,2,Based on the provided context and papers from ...,"Okay, based on the provided context and the in..."
8,40264093,3,Based on the context provided by the papers fr...,"Okay, here's an analysis of the provided text,..."
9,40264093,4,Individuals with Autism Spectrum Disorder (ASD...,"Okay, here's an analysis of the provided paper..."


## Synthetic questions

In [4]:
df_synth_q = pd.read_csv('../data/data-synth-question.csv', sep='\t', dtype=str)
df_synth_q

,pmid,ollama_seed,synthetic_question
0,40255413,0,Is gastrointestinal (GI) problems more common ...
1,40255413,1,Does autism spectrum disorders affect a child'...
2,40255413,2,What is one possible underlying factor contrib...
3,40255413,3,What is a common challenge that many individua...
4,40255413,4,How do gastrointestinal (GI) symptoms affect t...
...,...,...,...
495,40938164,0,Is autism a natural part of human diversity?
496,40938164,1,Can individuals with autism spectrum disorders...
497,40938164,2,What is it about individuals with autism spect...
498,40938164,3,What is the main difference between autism and...


## Abstracts

In [5]:
real_records = ingest.get_data_records('../data/data.csv')
df_abstr = pd.DataFrame.from_records(real_records)[['pmid', 'abstract']]
df_abstr

,pmid,abstract
0,40944767,Earlier identification of autistic traits is c...
1,40944766,This study examined friendship quality and the...
2,40944641,How do large-scale brain networks dynamically ...
3,40944186,Food selectivity is a prevalent and challengin...
4,40944170,Autism spectrum disorder (ASD) is a complex ne...
...,...,...
2995,40250148,To better understand medical comorbidity in pe...
2996,40249741,"Vaccines have saved millions of lives, yet the..."
2997,40249667,Genomic imprinting is an epigenetic phenomenon...
2998,40249409,Autistic children (AT) are known to exhibit di...


## Put together

In [6]:
df_together = df_synth_a.copy()
df_together = df_together.merge(df_synth_q, on=['pmid', 'ollama_seed'], how='left')
df_together = df_together.merge(df_abstr, on=['pmid'], how='left')
df_together

,pmid,ollama_seed,answer_llama3.2:1b,answer_gemma3:1b,synthetic_question,abstract
0,40255413,0,Based on the context provided by the papers fr...,"Based on the provided text, the answer is:\n\n...",Is gastrointestinal (GI) problems more common ...,Gastrointestinal (GI) symptoms are frequently ...
1,40255413,1,Based on the provided context and papers from ...,"Okay, here’s an answer based on the provided t...",Does autism spectrum disorders affect a child'...,Gastrointestinal (GI) symptoms are frequently ...
2,40255413,2,Based on the context provided by the papers fr...,"Okay, based on the provided context and papers...",What is one possible underlying factor contrib...,Gastrointestinal (GI) symptoms are frequently ...
3,40255413,3,"Based on the provided papers from PubMed, a co...","Okay, based on the provided context and papers...",What is a common challenge that many individua...,Gastrointestinal (GI) symptoms are frequently ...
4,40255413,4,Based on the provided context and papers from ...,"Okay, let’s analyze the provided text and answ...",How do gastrointestinal (GI) symptoms affect t...,Gastrointestinal (GI) symptoms are frequently ...
5,40264093,0,Based on the context provided by the papers yo...,"Okay, here’s an analysis of the provided text,...",Is there a link between autism and difficultie...,Autism Spectrum Disorder (ASD) affects social ...
6,40264093,1,Individuals on the autism spectrum can navigat...,"Okay, let’s answer your question based on the ...",How can individuals on the autism spectrum bes...,Autism Spectrum Disorder (ASD) affects social ...
7,40264093,2,Based on the provided context and papers from ...,"Okay, based on the provided context and the in...",Is Autism more likely to affect a person's abi...,Autism Spectrum Disorder (ASD) affects social ...
8,40264093,3,Based on the context provided by the papers fr...,"Okay, here's an analysis of the provided text,...",Is there a difference between the prevalence o...,Autism Spectrum Disorder (ASD) affects social ...
9,40264093,4,Individuals with Autism Spectrum Disorder (ASD...,"Okay, here's an analysis of the provided paper...",Can individuals with Autism Spectrum Disorder ...,Autism Spectrum Disorder (ASD) affects social ...


# Evaluation by cosine similarity

## Embed texts

In [7]:
# vectorizer handle
print(vectors.model_handle)

multi-qa-MiniLM-L6-cos-v1


In [8]:
# vector representations
print(datetime.now())
column_names_to_vectorize = ['answer_llama3.2:1b', 'answer_gemma3:1b', 'abstract']
vectorized = {name : vectors.model.encode(df_together[name].to_list()) \
for name in column_names_to_vectorize}
print(datetime.now())

2025-09-14 02:02:30.470221
2025-09-14 02:02:35.204101


## Compute cosine similarities

In [9]:
def get_cosine_similarities(model_handle):
    similarities = []
    for i in range(len(vectorized['abstract'])):
        vector_answer = vectorized['answer_'+model_handle][i]
        vector_abstract = vectorized['abstract'][i]
        similarity = vectors.model.similarity(vector_answer, vector_abstract).item()
        similarities.append(similarity)
    return similarities

In [10]:
# get similarities by model handle
df_similarities = pd.DataFrame({handle : get_cosine_similarities(handle) \
for handle in ['llama3.2:1b', 'gemma3:1b']})
df_similarities

,llama3.2:1b,gemma3:1b
0,0.825239,0.630141
1,0.500411,0.486283
2,0.920557,0.743348
3,0.354899,0.430685
4,0.842052,0.885504
5,0.567426,0.580948
6,0.604417,0.617429
7,0.774461,0.628251
8,0.812137,0.550834
9,0.763916,0.633520


## Compare models

In [11]:
# descriptive statistics, for llama3.2
df_similarities['llama3.2:1b'].describe()

count    30.000000
mean      0.652150
std       0.178849
min       0.277077
25%       0.572443
50%       0.658697
75%       0.811719
max       0.920557
Name: llama3.2:1b, dtype: float64

In [12]:
# descriptive statistics, for llama3.2
df_similarities['gemma3:1b'].describe()

count    30.000000
mean      0.612789
std       0.164969
min       0.163506
25%       0.558254
50%       0.626550
75%       0.733151
max       0.885504
Name: gemma3:1b, dtype: float64

In [13]:
# descriptive statistics, difference
(df_similarities['gemma3:1b']-df_similarities['llama3.2:1b']).describe()

count    30.000000
mean     -0.039361
std       0.083256
min      -0.261303
25%      -0.066257
50%      -0.026633
75%       0.008001
max       0.120719
dtype: float64

In [14]:
print(datetime.now())

2025-09-14 02:02:35.254281
